# Fig. 6C: Gene expression changes associated with Chaphamaparvovirus infection in chicken liver samples

Merge featureCounts and sample metadata for the diagnosed comparison in `Fig6C_DESeq2_diagnosed.R`.


In [1]:
import os
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


pandas: 2.3.1
numpy: 2.3.1


In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "Fig6C/counts"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}"

# Input files
input_folder_path = f"{basedir}/data/251215_rna_seq/revise_chaphama_featureCount"
metadata_folder_path = f"{basedir}/output/revise_final/Fig6B/group_analysis/sample_labels"

os.makedirs(path_out, exist_ok=True)
print(f"saving files to: {path_out}")


saving files to: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig6C/counts


In [3]:
# Read and merge featureCounts tables

meta_cols = [
    "Geneid",
    "Chr",
    "Start",
    "End",
    "Strand",
    "Length",
]

excluded_projects = {
    "PRJNA170656",
    "PRJNA498911",
}

count_files = sorted([
    os.path.join(input_folder_path, file_name)
    for file_name in os.listdir(input_folder_path)
    if (
        file_name.startswith("all_counts_")
        and file_name.endswith("_ver2.txt")
        and not any(
            project in file_name
            for project in excluded_projects
        )
    )
])

if not count_files:
    raise FileNotFoundError(
        f"Count fileが見つかりません: {input_folder_path}"
    )

count_tables = []

for count_file in count_files:
    table = pd.read_table(count_file)

    sample_cols = [
        column
        for column in table.columns
        if column not in meta_cols
    ]

    count_table = (
        table
        .set_index("Geneid")[sample_cols]
        .apply(pd.to_numeric)
    )

    count_tables.append(count_table)

    print(
        os.path.basename(count_file),
        count_table.shape,
    )

# Join on shared Geneid values
counts = pd.concat(
    count_tables,
    axis=1,
    join="inner",
    verify_integrity=True,
)

print()
print("Merged count matrix:", counts.shape)
display(counts.iloc[:5, :8])

all_counts_Chicken_Liver_ver2.txt (25466, 2)
all_counts_PRJNA577590_ver2.txt (25466, 20)
all_counts_PRJNA612882_ver2.txt (25466, 22)
all_counts_PRJNA622813_ver2.txt (25466, 8)

Merged count matrix: (25466, 52)


,Chap.miya,ChickenLiver1to3,SRR10293992,SRR10293993,SRR10293994,SRR10293995,SRR10293996,SRR10293997
Geneid,,,,,,,,
LOC124418406,0,0,2,0,2,0,0,2
LOC121106832,0,12,24,16,16,8,27,4
LOC107049418,0,30,6,4,3,18,8,6
LOC124417607,6,56,26,25,30,43,41,28
RNF212B,48,352,132,203,199,246,234,187


In [4]:
# Read sample metadata

metadata_files = sorted([
    os.path.join(metadata_folder_path, file_name)
    for file_name in os.listdir(metadata_folder_path)
    if (
        file_name.endswith(".csv")
        and not any(
            project in file_name
            for project in excluded_projects
        )
    )
])

if not metadata_files:
    raise FileNotFoundError(
        f"Metadata fileが見つかりません: {metadata_folder_path}"
    )

metadata = pd.concat(
    [
        pd.read_csv(metadata_file)
        for metadata_file in metadata_files
    ],
    ignore_index=True,
)

metadata = (
    metadata
    .drop_duplicates(subset="Run")
    .set_index("Run")
)

# Match the sample order in the count matrix
sample_metadata = (
    metadata[
        [
            "BioProject",
            "Positive",
            "Negative",
        ]
    ]
    .reindex(counts.columns)
    .copy()
)

# Convert strings and missing values to numeric values
sample_metadata["Positive"] = pd.to_numeric(
    sample_metadata["Positive"],
    errors="coerce",
)

sample_metadata["Negative"] = pd.to_numeric(
    sample_metadata["Negative"],
    errors="coerce",
)

# Keep samples with Positive == 1 or Negative == 1
sample_metadata["Virus_status"] = np.select(
    [
        sample_metadata["Positive"].eq(1),
        sample_metadata["Negative"].eq(1),
    ],
    [
        "Positive",
        "Negative",
    ],
    default="Exclude",
)

# Add Chap.miya as a positive sample
sample_metadata.loc[
    "Chap.miya",
    [
        "BioProject",
        "Positive",
        "Negative",
        "Virus_status",
    ],
] = [
    "Chicken_Liver",
    1,
    0,
    "Positive",
]

# Add ChickenLiver1to3 as a positive sample
sample_metadata.loc[
    "ChickenLiver1to3",
    [
            "BioProject",
            "Positive",
            "Negative",
            "Virus_status",
        ],
    ] = [
        "Chicken_Liver",
        1,
        0,
        "Positive",
]

# Select samples for analysis
analysis_samples = [
    sample
    for sample in counts.columns
    if sample_metadata.loc[sample, "Virus_status"]
    in {"Positive", "Negative"}
]

print("Samples used for analysis:", len(analysis_samples))


Samples used for analysis: 48


In [5]:
# Save tables
counts.loc[:, analysis_samples].to_csv(
    f"{path_out}/merged_four_project_all_counts_ver2.txt",
    sep="\t",
)

sample_metadata.loc[analysis_samples].to_csv(
    f"{path_out}/sample_metadata.txt",
    sep="\t",
    index_label="Run",
)